# 🐝 Embodied Hornet — Spiking SLAM + Neuromechanical Flight Control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lhooz/embodied_hornet/blob/main/notebooks/demo_colab.ipynb)

This notebook demonstrates the **Embodied Hornet** framework: a biologically-grounded autonomous micro-aerial vehicle that combines:

| Layer | What it does |
|:---|:---|
| **Neuromechanical hover specialist** | Pre-trained IDA-PBC ICNN that keeps the hornet aloft using passivity-preserving control |
| **Spiking SLAM system** | Event-camera + 3-beam ToF sensor fusion for online mapping |
| **SHAC navigation policy** | Short-Horizon Actor-Critic trained with DNAG (Dynamic Neural Attention Gate) that learns to navigate obstacle rooms |

---
**Pipeline:**
1. ⚙️ **Setup** — Clone repo (with submodules), install deps
2. 🦟 **Phase 1** *(optional)* — Train the hover specialist with hornetRL
3. 🗺️ **Phase 2** — Train SHAC navigation on the 1 m × 1 m obstacle arena
4. 🎬 **Replay** — Visualize the trained policy with SLAM sensor overlay

> 💡 **Tip:** Switch Runtime → GPU for ~10× faster training. The hover specialist (`hover_params.pkl`) is included in the repo so you can skip Phase 1.

In [ ]:
# ============================================================
# CELL 1 — Mount Google Drive + Clone Repository
# ============================================================
from google.colab import drive
import os

print("→ Mounting Google Drive for checkpoint persistence...")
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/EmbodiedHornet_Dev'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)

REPO_NAME = 'embodied_hornet'
REPO_URL  = 'https://github.com/lhooz/embodied_hornet.git'

if not os.path.exists(REPO_NAME):
    print("→ Cloning embodied_hornet (with submodules)...")
    !git clone --recurse-submodules {REPO_URL}
else:
    print("→ Repo already exists. Pulling latest...")
    os.chdir(REPO_NAME)
    !git pull
    os.chdir('..')

REPO_PATH  = os.path.join(PROJECT_ROOT, REPO_NAME)
HORNET_SRC = os.path.join(REPO_PATH, 'hornetRL')
SLAM_SRC   = os.path.join(REPO_PATH, 'neuro-symbolic-slam', 'src')
os.chdir(REPO_PATH)

# Force-update all submodules (repairs incomplete checkouts)
print("→ Updating submodules...")
!git submodule update --init --recursive --force

# Verify actual files exist (not just directories)
checks = [
    ('hornetRL/hornetRL/fly_system.py',          os.path.join(HORNET_SRC, 'hornetRL', 'fly_system.py')),
    ('neuro-symbolic-slam/src/sparse_forest.py', os.path.join(SLAM_SRC, 'sparse_forest.py')),
    ('embodied_hornet/hover_params.pkl',         os.path.join(REPO_PATH, 'embodied_hornet', 'hover_params.pkl')),
]
all_ok = True
for label, path in checks:
    ok = os.path.exists(path)
    print(f"   {'✅' if ok else '❌'} {label}")
    if not ok:
        all_ok = False

if not all_ok:
    print("\n⚠️  Some files missing. Attempting targeted re-checkout...")
    !git -C "{REPO_PATH}" submodule update --force --checkout hornetRL
    !git -C "{REPO_PATH}" submodule update --force --checkout neuro-symbolic-slam
    print("   Recheck:")
    for label, path in checks:
        print(f"   {'✅' if os.path.exists(path) else '❌ STILL MISSING'} {label}")

print(f"\n→ Repo ready at: {REPO_PATH}")

In [ ]:
# ============================================================
# CELL 2 — Install Dependencies & Wire Imports
# ============================================================
import os, sys, subprocess, importlib

if 'REPO_PATH' not in dir():
    REPO_PATH  = '/content/drive/MyDrive/EmbodiedHornet_Dev/embodied_hornet'
    HORNET_SRC = os.path.join(REPO_PATH, 'hornetRL')
    SLAM_SRC   = os.path.join(REPO_PATH, 'neuro-symbolic-slam', 'src')
os.chdir(REPO_PATH)

# ── 1. Install Python dependencies ────────────────────────
print("→ Installing Python dependencies...")
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("   ✅ pip install -e . succeeded")
else:
    print("   ⚠️  Editable install failed — installing deps directly")
    for dep in ['dm-haiku>=0.0.12', 'optax>=0.2.0', 'imageio>=2.34.0']:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', dep, '-q'],
                           capture_output=True, text=True)
        print(f"   {'✅' if r.returncode == 0 else '❌'} {dep}")

# ── 2. Register submodule paths in sys.path ───────────────
# Remove stale entries first, then re-add — this ensures the
# path finders re-scan directories whose contents changed
# (e.g. submodule was empty on first run, populated now).
print("\n→ Registering submodule paths in sys.path...")
for label, _p in [('hornetRL (repo root)',    HORNET_SRC),
                  ('neuro-symbolic-slam/src', SLAM_SRC),
                  ('repo root',               REPO_PATH)]:
    if not os.path.isdir(_p):
        print(f"   ❌ missing: {label} — rerun Cell 1")
        continue
    # Remove any existing entry so we get a fresh finder
    while _p in sys.path:
        sys.path.remove(_p)
    sys.path.insert(0, _p)
    print(f"   ✓ {label}")

# Flush all cached module lookups — critical when paths were
# previously added while submodule dirs were still empty.
sys.path_importer_cache.clear()
importlib.invalidate_caches()
print("   ✓ importlib and path finder caches cleared")

# ── 3. Verify source files exist ──────────────────────────
print("\n→ Verifying source files...")
for fname, fpath in [
    ('sparse_forest.py',   os.path.join(SLAM_SRC,   'sparse_forest.py')),
    ('snn_slam_system.py', os.path.join(SLAM_SRC,   'snn_slam_system.py')),
    ('fly_system.py',      os.path.join(HORNET_SRC, 'hornetRL', 'fly_system.py')),
]:
    print(f"   {'✅' if os.path.exists(fpath) else '❌ MISSING'} {fname}")

# ── 4. Import smoke-test ──────────────────────────────────
print("\n→ Verifying imports...")
try:
    import jax
    import haiku as hk
    import optax
    from hornetRL.fly_system import FlappingFlySystem       # noqa
    from hornetRL.fluid_surrogate import JaxSurrogateEngine # noqa
    from sparse_forest import generate_obstacles            # noqa
    from snn_slam_system import SNNSLAMSystem               # noqa
    print("   ✅ All imports OK — ready to train")
except ImportError as e:
    print(f"   ❌ Import error: {e}")
    print(f"   sys.path (first 10):")
    for p in sys.path[:10]:
        print(f"      {p}")
    print(f"   Try: Runtime → Restart runtime, then rerun all cells")

# ── 5. GPU check ──────────────────────────────────────────
devices = jax.devices()
print(f"\n→ JAX devices: {devices}")
HAS_GPU = any('gpu' in str(d).lower() or 'cuda' in str(d).lower() for d in devices)
print(f"→ GPU: {'✅ Yes' if HAS_GPU else '❌ No — Runtime → Change runtime type → T4 GPU'}")

In [ ]:
# ============================================================
# CELL 3 — Configuration
# ============================================================
import os
if 'REPO_PATH' not in dir():
    REPO_PATH  = '/content/drive/MyDrive/EmbodiedHornet_Dev/embodied_hornet'
    HORNET_SRC = os.path.join(REPO_PATH, 'hornetRL')
    SLAM_SRC   = os.path.join(REPO_PATH, 'neuro-symbolic-slam', 'src')
os.chdir(REPO_PATH)

# ── User-configurable ──────────────────────────────────────
USE_GPU           = True
PHASE1_STEPS      = 2000
PHASE2_STEPS      = 100000
QUICK_DEMO_STEPS  = 500
# ───────────────────────────────────────────────────────────

CKPT_DIR  = os.path.join(REPO_PATH, 'checkpoints_shac')
HOVER_PKL = os.path.join(REPO_PATH, 'embodied_hornet', 'hover_params.pkl')
GPU_FLAG  = '--gpu' if USE_GPU else ''

os.makedirs(CKPT_DIR, exist_ok=True)

print("=" * 50)
print("        Embodied Hornet — Demo Config")
print("=" * 50)
print(f"  Repo path     : {REPO_PATH}")
print(f"  Checkpoint dir: {CKPT_DIR}")
print(f"  GPU mode      : {USE_GPU}")
print(f"  Hover PKL     : {'✅ found' if os.path.exists(HOVER_PKL) else '❌ missing (Phase 1 needed)'}")

import glob, re
existing = sorted(glob.glob(os.path.join(CKPT_DIR, 'shac_params_*.pkl')),
                  key=lambda f: int(re.search(r'_(\d+)', f).group(1)))
if existing:
    step = int(re.search(r'_(\d+)', existing[-1]).group(1))
    print(f"  Latest ckpt   : {os.path.basename(existing[-1])} (step {step}) — will resume")
else:
    print("  Latest ckpt   : none — will train from scratch")

In [ ]:
# ============================================================
# CELL 4 — Phase 1: Hover Specialist (skip if already present)
# ============================================================
import os, shutil

if os.path.exists(HOVER_PKL):
    print(f"✅ hover_params.pkl already present — skipping Phase 1.")
    print(f"   Path: {HOVER_PKL}")
else:
    print("⚠️  hover_params.pkl not found. Running Phase 1 hover training...")
    print(f"   This will train for {PHASE1_STEPS} steps (~30 min on T4 GPU).\n")

    HOVER_CKPT_DIR = os.path.join(REPO_PATH, 'hornetRL', 'checkpoints_shac')
    os.makedirs(HOVER_CKPT_DIR, exist_ok=True)
    os.environ['PYTHONPATH'] = f"{REPO_PATH}:{HORNET_SRC}:{SLAM_SRC}"

    !python -u -m hornetRL.train {GPU_FLAG} \
        --dir "{HOVER_CKPT_DIR}" \
        --steps {PHASE1_STEPS}

    hover_src = os.path.join(HOVER_CKPT_DIR, 'hover_params.pkl')
    if os.path.exists(hover_src):
        shutil.copy(hover_src, HOVER_PKL)
        print(f"\n✅ hover_params.pkl copied to: {HOVER_PKL}")
    else:
        print("\n❌ Phase 1 did not produce hover_params.pkl. Check logs above.")

In [ ]:
# ============================================================
# CELL 5 — Phase 2: Navigation Training (SHAC + SLAM + DNAG)
# ============================================================
# PYTHONPATH is set so the subprocess finds submodule packages.
# Interrupt (⏹) to pause — re-run this cell to resume.
# ============================================================
import os

if 'REPO_PATH' not in dir():
    REPO_PATH  = '/content/drive/MyDrive/EmbodiedHornet_Dev/embodied_hornet'
    HORNET_SRC = os.path.join(REPO_PATH, 'hornetRL')
    SLAM_SRC   = os.path.join(REPO_PATH, 'neuro-symbolic-slam', 'src')
    CKPT_DIR   = os.path.join(REPO_PATH, 'checkpoints_shac')
    USE_GPU    = True
    GPU_FLAG   = '--gpu'

os.chdir(REPO_PATH)
os.environ['PYTHONPATH'] = f"{REPO_PATH}:{HORNET_SRC}:{SLAM_SRC}"

print("→ Starting Phase 2: Navigation Training")
print(f"   Checkpoint: {CKPT_DIR}")
print(f"   GPU mode  : {USE_GPU}")
print(f"   PYTHONPATH set ✓")
print()

!python -u -m embodied_hornet.train {GPU_FLAG} 2>&1 | tee /tmp/train_out.txt

In [ ]:
# ============================================================
# CELL 5b — Quick Demo Run (≈ 5 min)
# ============================================================
# Uncomment and run instead of Cell 5 for a quick sanity check.
# ============================================================

# import os
# if 'REPO_PATH' not in dir():
#     REPO_PATH  = '/content/drive/MyDrive/EmbodiedHornet_Dev/embodied_hornet'
#     HORNET_SRC = os.path.join(REPO_PATH, 'hornetRL')
#     SLAM_SRC   = os.path.join(REPO_PATH, 'neuro-symbolic-slam', 'src')
#     QUICK_DEMO_STEPS = 500
#     GPU_FLAG = '--gpu'
# os.chdir(REPO_PATH)
# os.environ['PYTHONPATH'] = f"{REPO_PATH}:{HORNET_SRC}:{SLAM_SRC}"
# !python -u -m embodied_hornet.train {GPU_FLAG} 2>&1 | tee /tmp/train_out.txt

print("(Quick demo cell — uncomment to use)")

In [ ]:
# ============================================================
# CELL 6 — Replay: Display Latest Epoch GIF
# ============================================================
import os, glob, re, shutil
from IPython.display import Image, display

if 'REPO_PATH' not in dir():
    REPO_PATH = '/content/drive/MyDrive/EmbodiedHornet_Dev/embodied_hornet'
if 'CKPT_DIR' not in dir():
    CKPT_DIR = os.path.join(REPO_PATH, 'checkpoints_shac')

gifs = sorted(
    glob.glob(os.path.join(CKPT_DIR, 'epoch_*.gif')) +
    glob.glob(os.path.join(REPO_PATH, 'epoch_*.gif')),
    key=lambda f: int(re.search(r'epoch_(\d+)', f).group(1))
)

if not gifs:
    print("⚠️  No epoch GIFs found yet. Run the training cell first.")
else:
    latest_gif = gifs[-1]
    step = int(re.search(r'epoch_(\d+)', latest_gif).group(1))
    tmp_gif = f'/tmp/replay_step_{step}.gif'
    shutil.copy(latest_gif, tmp_gif)
    print(f"{'='*60}")
    print(f"  SLAM Navigation Room (left) + Wing Mechanics (right)")
    print(f"  Step {step} | ToF beams: blue=left/right, yellow=center")
    print(f"{'='*60}")
    display(Image(filename=tmp_gif, width=900))
    if len(gifs) > 1:
        print(f"\n→ All epoch steps: ",
              [int(re.search(r'epoch_(\d+)', g).group(1)) for g in gifs])

In [ ]:
# ============================================================
# CELL 7 — Training Progress Summary
# ============================================================
import re, os
import matplotlib.pyplot as plt

LOG_FILE = '/tmp/train_out.txt'

if not os.path.exists(LOG_FILE):
    print(f"Log not found at {LOG_FILE}.")
    print("Cell 5 writes it via '| tee /tmp/train_out.txt'.")
else:
    steps, losses, rewards, frcs = [], [], [], []
    pat = re.compile(r'Step (\d+).*Loss: ([\d.e+\-]+) .*Rew: ([\d.\-]+).*Frc:([\d.]+)')
    with open(LOG_FILE) as f:
        for line in f:
            m = pat.search(line)
            if m:
                steps.append(int(m.group(1)))
                losses.append(float(m.group(2)))
                rewards.append(float(m.group(3)))
                frcs.append(float(m.group(4)))
    if not steps:
        print("No Step log lines found yet.")
    else:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.patch.set_facecolor('#0d0d0d')
        for ax in axes:
            ax.set_facecolor('#111111')
            ax.tick_params(colors='#aaaaaa')
            for sp in ax.spines.values(): sp.set_edgecolor('#444444')
        axes[0].plot(steps, losses,  color='#ff6666', lw=1.5)
        axes[0].set_title('Total Loss',        color='white')
        axes[0].set_xlabel('Step',             color='#aaaaaa')
        axes[1].plot(steps, rewards, color='#66ff99', lw=1.5)
        axes[1].set_title('Mean Reward',       color='white')
        axes[1].set_xlabel('Step',             color='#aaaaaa')
        axes[2].plot(steps, frcs,    color='#ffaa44', lw=1.5)
        axes[2].set_title('Force Match Error', color='white')
        axes[2].set_xlabel('Step',             color='#aaaaaa')
        axes[2].set_yscale('log')
        plt.suptitle(f'Embodied Hornet — Training ({len(steps)} steps)',
                     color='white', fontsize=13)
        plt.tight_layout()
        plt.show()
        print(f"Steps: {len(steps)} | Best reward: {max(rewards):.1f} | "
              f"Frc: {frcs[0]:.4f} → {frcs[-1]:.4f} "
              f"({'✅ improved' if frcs[-1] < frcs[0] else '⏳ not yet'})")